# Stage 2: Joint NER + Sentiment Fine-tuning (Resume)

Resumes from Stage 1 checkpoint saved on Drive.

**Stage 1 Results (completed):**
- 5 epochs, ~19 hours on A100 80GB
- Best val NER loss: 130.05 (epoch 5)
- Best NER F1: 0.7612

**Stage 2 Plan:**
- Load Stage 1 best weights
- Joint NER + Sentiment training with curriculum learning
- 5 epochs, lr=1e-5

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Project path + install deps
import os
import sys

PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
assert os.path.exists(PROJECT_PATH), f"Project not found: {PROJECT_PATH}"

!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf accelerate

# HF auth
from getpass import getpass
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('Found HF_TOKEN in Colab Secrets')
except (ImportError, userdata.SecretNotFoundError):
    hf_token = getpass('Paste your HF token (or Enter to skip): ').strip()
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('Authenticated with HF Hub')

# Add project to path
sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# 4. Imports
import gc
import time
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

from training.preprocessing import DataPreprocessor
from training.dataset import create_data_loaders
from training.trainer import compute_ner_metrics, compute_sentiment_metrics
from models.pipeline import FinancialEntitySentimentModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# 5. Configuration
TRAIN_FILE = f"{PROJECT_PATH}/data/labeled/final/train.jsonl"
VAL_FILE = f"{PROJECT_PATH}/data/labeled/final/val.jsonl"
HOLDOUT_FILE = f"{PROJECT_PATH}/data/labeled/final/holdout.jsonl"

CONFIG = {
    "train_file": TRAIN_FILE,
    "val_file": VAL_FILE,
    "holdout_file": HOLDOUT_FILE,
    "encoder_name": "allenai/longformer-large-4096",
    "hidden_size": 1024,
    "max_length": 2048,
    "use_crf": True,
    "batch_size": 5,
    "gradient_accumulation": 2,  # effective batch = 10
    "stage2_epochs": 5,
    "stage2_lr": 1e-5,
    "stage1_checkpoint_dir": f"{PROJECT_PATH}/checkpoints/stage1_ner_large",
    "stage2_checkpoint_dir": f"{PROJECT_PATH}/checkpoints/stage2_joint_large",
}

os.makedirs(CONFIG["stage2_checkpoint_dir"], exist_ok=True)

# Verify Stage 1 checkpoint exists
stage1_ckpt = f"{CONFIG['stage1_checkpoint_dir']}/best_model.pt"
assert os.path.exists(stage1_ckpt), f"Stage 1 checkpoint not found: {stage1_ckpt}"
print(f"Stage 1 checkpoint: {stage1_ckpt}")
!ls -lh "{CONFIG['stage1_checkpoint_dir']}"

print(f"\nEffective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation']}")

In [ ]:
# 6. Load Stage 1 history (for final visualization)
last_epoch_ckpt = f"{CONFIG['stage1_checkpoint_dir']}/checkpoint_epoch_5.pt"
if os.path.exists(last_epoch_ckpt):
    ckpt = torch.load(last_epoch_ckpt, map_location='cpu', weights_only=False)
    history_stage1 = ckpt.get('history', {'train_loss': [], 'val_loss': [], 'val_ner_f1': []})
    best_val_loss_s1 = min(history_stage1['val_loss']) if history_stage1['val_loss'] else float('inf')
    del ckpt
    print('Stage 1 training history:')
    for i, (tl, vl, f1) in enumerate(zip(
        history_stage1['train_loss'],
        history_stage1['val_loss'],
        history_stage1['val_ner_f1']
    )):
        print(f'  Epoch {i+1}: train_loss={tl:.4f}, val_loss={vl:.4f}, ner_f1={f1:.4f}')
    print(f'\nBest val loss: {best_val_loss_s1:.4f}, Best F1: {max(history_stage1["val_ner_f1"]):.4f}')
else:
    print('WARNING: Stage 1 history checkpoint not found')
    history_stage1 = {'train_loss': [], 'val_loss': [], 'val_ner_f1': []}
    best_val_loss_s1 = float('inf')

In [ ]:
# 7. Load data
preprocessor = DataPreprocessor(
    model_name=CONFIG['encoder_name'],
    max_length=CONFIG['max_length'],
)

print('Loading and preprocessing data...')
train_loader, val_loader = create_data_loaders(
    train_files=CONFIG['train_file'],
    val_files=CONFIG['val_file'],
    preprocessor=preprocessor,
    batch_size=CONFIG['batch_size'],
)

print(f'\nTrain batches: {len(train_loader)} (batch_size={CONFIG["batch_size"]})')
print(f'Val batches: {len(val_loader)}')

In [ ]:
# 8. Define training functions

def train_epoch_with_accumulation(
    model, train_loader, optimizer, scheduler, scaler, device,
    accumulation_steps, ner_weight=1.0, sentiment_weight=0.0, gradient_clip=1.0,
):
    """Train one epoch with gradient accumulation."""
    model.train()
    total_loss = 0.0
    total_ner_loss = 0.0
    total_sentiment_loss = 0.0
    num_batches = 0

    ner_criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
    sentiment_criterion = nn.MSELoss(reduction='none')
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc='Training')
    for step, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ner_labels = batch['ner_labels'].to(device)
        entity_masks = batch['entity_masks'].to(device)
        sentiment_targets = batch['sentiment_scores'].to(device)
        entity_mask_valid = batch['entity_mask_valid'].to(device)

        with torch.amp.autocast(device_type='cuda'):
            ner_output, sentiment_preds = model(
                input_ids=input_ids, attention_mask=attention_mask,
                entity_masks=entity_masks, ner_labels=ner_labels,
            )
            if isinstance(ner_output, dict):
                ner_loss = ner_output.get('loss', torch.tensor(0.0, device=device))
            else:
                bs, sl, nl = ner_output.shape
                ner_loss = ner_criterion(ner_output.view(-1, nl), ner_labels.view(-1))

            sentiment_loss_all = sentiment_criterion(sentiment_preds, sentiment_targets)
            sentiment_loss_masked = sentiment_loss_all * entity_mask_valid
            num_valid = entity_mask_valid.sum()
            sentiment_loss = sentiment_loss_masked.sum() / num_valid if num_valid > 0 else torch.tensor(0.0, device=device)

            loss = (ner_weight * ner_loss + sentiment_weight * sentiment_loss) / accumulation_steps

        scaler.scale(loss).backward()
        total_loss += loss.item() * accumulation_steps
        total_ner_loss += ner_loss.item()
        total_sentiment_loss += sentiment_loss.item() if num_valid > 0 else 0
        num_batches += 1

        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix({
                'loss': f'{total_loss/num_batches:.4f}',
                'ner': f'{total_ner_loss/num_batches:.4f}',
                'sent': f'{total_sentiment_loss/num_batches:.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}'
            })

    if (step + 1) % accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    return {
        'train_loss': total_loss / num_batches,
        'train_ner_loss': total_ner_loss / num_batches,
        'train_sentiment_loss': total_sentiment_loss / num_batches,
    }


@torch.no_grad()
def evaluate_model(model, val_loader, device):
    """Evaluate on validation set."""
    model.eval()
    ner_criterion = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
    sentiment_criterion = nn.MSELoss(reduction='none')

    total_ner_loss = 0.0
    total_sentiment_loss = 0.0
    total_samples = 0
    total_entities = 0
    all_ner_preds, all_ner_labels = [], []
    all_sentiment_preds, all_sentiment_targets = [], []

    for batch in tqdm(val_loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ner_labels = batch['ner_labels'].to(device)
        entity_masks = batch['entity_masks'].to(device)
        sentiment_targets = batch['sentiment_scores'].to(device)
        entity_mask_valid = batch['entity_mask_valid'].to(device)
        batch_size = input_ids.shape[0]

        with torch.amp.autocast(device_type='cuda'):
            ner_output, sentiment_preds = model(
                input_ids=input_ids, attention_mask=attention_mask,
                entity_masks=entity_masks,
            )

        if isinstance(ner_output, dict):
            ner_logits = ner_output['logits']
            if hasattr(model.ner_head, 'crf'):
                ner_loss = -model.ner_head.crf(ner_logits, ner_labels, mask=attention_mask.bool(), reduction='mean')
            else:
                _, sl, nl = ner_logits.shape
                ner_loss = ner_criterion(ner_logits.view(-1, nl), ner_labels.view(-1))
        else:
            ner_logits = ner_output
            _, sl, nl = ner_logits.shape
            ner_loss = ner_criterion(ner_logits.view(-1, nl), ner_labels.view(-1))
        total_ner_loss += ner_loss.item() * batch_size

        sentiment_loss_all = sentiment_criterion(sentiment_preds, sentiment_targets)
        sentiment_loss_masked = sentiment_loss_all * entity_mask_valid
        num_valid = entity_mask_valid.sum().item()
        if num_valid > 0:
            total_sentiment_loss += sentiment_loss_masked.sum().item()
            total_entities += num_valid
        total_samples += batch_size

        if isinstance(ner_output, dict) and 'predictions' in ner_output:
            ner_preds = ner_output['predictions']
        else:
            ner_preds = ner_logits.argmax(dim=-1)

        valid_mask = attention_mask.bool()
        for i in range(batch_size):
            all_ner_preds.extend(ner_preds[i][valid_mask[i]].cpu().tolist())
            all_ner_labels.extend(ner_labels[i][valid_mask[i]].cpu().tolist())
        for i in range(batch_size):
            for j in range(entity_mask_valid.shape[1]):
                if entity_mask_valid[i, j] > 0:
                    all_sentiment_preds.append(sentiment_preds[i, j].item())
                    all_sentiment_targets.append(sentiment_targets[i, j].item())

    metrics = {
        'val_ner_loss': total_ner_loss / total_samples,
        'val_sentiment_loss': total_sentiment_loss / max(total_entities, 1),
        'val_total_loss': total_ner_loss / total_samples + 0.5 * total_sentiment_loss / max(total_entities, 1),
    }
    metrics.update(compute_ner_metrics(all_ner_preds, all_ner_labels))
    if all_sentiment_preds:
        metrics.update(compute_sentiment_metrics(all_sentiment_preds, all_sentiment_targets))
    return metrics

print('Training functions defined.')

## Stage 2: Joint NER + Sentiment Fine-tuning

In [ ]:
# 9. Create model and load Stage 1 weights
print('=' * 60)
print('STAGE 2: JOINT NER + SENTIMENT FINE-TUNING')
print('=' * 60)

LOG_FILE = f"{PROJECT_PATH}/logs/stage2_large_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
os.makedirs(f'{PROJECT_PATH}/logs', exist_ok=True)

def log_message(msg):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(LOG_FILE, 'a') as f:
        f.write(f'{ts} - {msg}\n')
    print(msg)

model = FinancialEntitySentimentModel(
    encoder_name=CONFIG['encoder_name'],
    hidden_size=CONFIG['hidden_size'],
    use_crf_ner=CONFIG['use_crf'],
)
model = model.to(device)

# Load Stage 1 best weights
stage1_ckpt_path = f"{CONFIG['stage1_checkpoint_dir']}/best_model.pt"
log_message(f'Loading Stage 1 weights: {stage1_ckpt_path}')
checkpoint = torch.load(stage1_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
log_message(f"Stage 1 weights loaded (from epoch {checkpoint.get('epoch', -1) + 1})")
del checkpoint
torch.cuda.empty_cache()

# Optimizer, scheduler, scaler
optimizer = AdamW(model.parameters(), lr=CONFIG['stage2_lr'], weight_decay=0.01)
num_steps = len(train_loader) * CONFIG['stage2_epochs']
scheduler = CosineAnnealingLR(optimizer, T_max=num_steps)
scaler = torch.amp.GradScaler()

log_message(f"Config: batch={CONFIG['batch_size']}x{CONFIG['gradient_accumulation']}, lr={CONFIG['stage2_lr']}, epochs={CONFIG['stage2_epochs']}")
log_message(f'Log: {LOG_FILE}')
if torch.cuda.is_available():
    log_message(f'GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# 10. Stage 2 training loop with curriculum learning
log_message('Starting Stage 2 training...')
start_time = time.time()

best_val_loss = float('inf')
history_stage2 = {'train_loss': [], 'val_loss': [], 'val_ner_f1': [], 'val_sentiment_mse': []}
accumulation_steps = CONFIG['gradient_accumulation']

for epoch in range(CONFIG['stage2_epochs']):
    epoch_start = time.time()

    # Curriculum: NER weight 1.0->0.5, sentiment weight 0.3->1.0
    progress = epoch / max(CONFIG['stage2_epochs'] - 1, 1)
    ner_weight = 1.0 + (0.5 - 1.0) * progress
    sentiment_weight = 0.3 + (1.0 - 0.3) * progress

    log_message(f'Epoch {epoch+1}/{CONFIG["stage2_epochs"]} - NER w: {ner_weight:.2f}, Sent w: {sentiment_weight:.2f}')

    train_metrics = train_epoch_with_accumulation(
        model=model, train_loader=train_loader, optimizer=optimizer,
        scheduler=scheduler, scaler=scaler, device=device,
        accumulation_steps=accumulation_steps,
        ner_weight=ner_weight, sentiment_weight=sentiment_weight,
    )

    val_metrics = evaluate_model(model, val_loader, device)

    epoch_time = time.time() - epoch_start
    gpu_mem = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0

    log_message(
        f'Epoch {epoch+1}: train_loss={train_metrics["train_loss"]:.4f}, '
        f'val_loss={val_metrics["val_total_loss"]:.4f}, '
        f'ner_f1={val_metrics.get("ner_f1", 0):.4f}, '
        f'sent_mse={val_metrics.get("sentiment_mse", 0):.4f}, '
        f'time={epoch_time/60:.1f}min, gpu_peak={gpu_mem:.1f}GB'
    )

    history_stage2['train_loss'].append(train_metrics['train_loss'])
    history_stage2['val_loss'].append(val_metrics['val_total_loss'])
    history_stage2['val_ner_f1'].append(val_metrics.get('ner_f1', 0))
    history_stage2['val_sentiment_mse'].append(val_metrics.get('sentiment_mse', 0))

    if val_metrics['val_total_loss'] < best_val_loss:
        best_val_loss = val_metrics['val_total_loss']
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
            'epoch': epoch,
        }, f"{CONFIG['stage2_checkpoint_dir']}/best_model.pt")
        log_message(f'  -> New best model! (val_loss={best_val_loss:.4f})')

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'history': history_stage2,
    }, f"{CONFIG['stage2_checkpoint_dir']}/checkpoint_epoch_{epoch+1}.pt")

    elapsed = time.time() - start_time
    remaining = (elapsed / (epoch + 1)) * (CONFIG['stage2_epochs'] - (epoch + 1))
    log_message(f'  -> ETA: {remaining/60:.1f} min remaining')

elapsed = time.time() - start_time
log_message('=' * 60)
log_message('STAGE 2 COMPLETE!')
log_message(f'Total time: {elapsed/60:.1f} min ({elapsed/3600:.2f} hours)')
log_message(f'Best val loss: {best_val_loss:.4f}')
log_message(f'Best NER F1: {max(history_stage2["val_ner_f1"]):.4f}')
log_message(f'Best Sentiment MSE: {min(history_stage2["val_sentiment_mse"]):.4f}')
log_message('=' * 60)

## Training Summary

In [ ]:
# 11. Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

ax = axes[0, 0]
if history_stage1['train_loss']:
    ax.plot(history_stage1['train_loss'], label='Train')
    ax.plot(history_stage1['val_loss'], label='Val')
ax.set_title('Stage 1: NER Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()

ax = axes[0, 1]
if history_stage1['val_ner_f1']:
    ax.plot(history_stage1['val_ner_f1'], 'g-', label='NER F1')
ax.set_title('Stage 1: NER F1')
ax.set_xlabel('Epoch'); ax.set_ylabel('F1'); ax.legend()

ax = axes[0, 2]
ax.plot(history_stage2['train_loss'], label='Train')
ax.plot(history_stage2['val_loss'], label='Val')
ax.set_title('Stage 2: Joint Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend()

ax = axes[1, 0]
ax.plot(history_stage2['val_ner_f1'], 'g-', label='NER F1')
ax.set_title('Stage 2: NER F1')
ax.set_xlabel('Epoch'); ax.set_ylabel('F1'); ax.legend()

ax = axes[1, 1]
ax.plot(history_stage2['val_sentiment_mse'], 'r-', label='Sentiment MSE')
ax.set_title('Stage 2: Sentiment MSE')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE'); ax.legend()

ax = axes[1, 2]
ax.axis('off')
s1_f1 = max(history_stage1['val_ner_f1']) if history_stage1['val_ner_f1'] else 0
summary = f"""
TRAINING SUMMARY (Longformer-Large)
====================================
Stage 1 (NER only): 5 epochs
  Best NER F1: {s1_f1:.4f}
  Best Val Loss: {best_val_loss_s1:.4f}

Stage 2 (Joint): {CONFIG['stage2_epochs']} epochs
  Best NER F1: {max(history_stage2['val_ner_f1']):.4f}
  Best Sent MSE: {min(history_stage2['val_sentiment_mse']):.4f}
  Best Val Loss: {best_val_loss:.4f}
"""
ax.text(0.05, 0.5, summary, fontsize=11, family='monospace', verticalalignment='center', transform=ax.transAxes)

plt.tight_layout()
os.makedirs(f'{PROJECT_PATH}/outputs', exist_ok=True)
plt.savefig(f'{PROJECT_PATH}/outputs/training_curves_large.png', dpi=150)
plt.show()
print(f'Saved: {PROJECT_PATH}/outputs/training_curves_large.png')

In [ ]:
# 12. Final model info
print('=' * 60)
print('TRAINING COMPLETE!')
print('=' * 60)
print(f'\nCheckpoints:')
print(f'  Stage 1: {CONFIG["stage1_checkpoint_dir"]}/best_model.pt')
print(f'  Stage 2: {CONFIG["stage2_checkpoint_dir"]}/best_model.pt')
print()
!ls -lh "{CONFIG['stage2_checkpoint_dir']}"

## (Optional) Holdout Evaluation

In [ ]:
# 13. Evaluate on holdout set
if os.path.exists(CONFIG['holdout_file']):
    print('Loading holdout set...')
    holdout_loader, _ = create_data_loaders(
        train_files=CONFIG['holdout_file'],
        preprocessor=preprocessor,
        batch_size=CONFIG['batch_size'],
    )
    holdout_metrics = evaluate_model(model, holdout_loader, device)

    print('\n' + '=' * 60)
    print('HOLDOUT SET EVALUATION (9,371 articles)')
    print('=' * 60)
    print(f'  NER F1:          {holdout_metrics.get("ner_f1", 0):.4f}')
    print(f'  NER Precision:   {holdout_metrics.get("ner_precision", 0):.4f}')
    print(f'  NER Recall:      {holdout_metrics.get("ner_recall", 0):.4f}')
    print(f'  Sentiment MSE:   {holdout_metrics.get("sentiment_mse", 0):.4f}')
    print(f'  Sentiment MAE:   {holdout_metrics.get("sentiment_mae", 0):.4f}')
else:
    print(f'Holdout file not found: {CONFIG["holdout_file"]}')

In [ ]:
# Disconnect when done
# from google.colab import runtime
# runtime.unassign()